In [32]:
import pandas as pd

# data_middle.csv 파일 불러옴
data_middle = pd.read_csv(
    r"..\..\data\processed\data_middle.csv",
    index_col=0,
    parse_dates=True
)

# Signal1: MA5와 MA15의 골든크로스와 데드크로스
data_middle['Signal1'] = 'Stay'

buy_cond = (data_middle['KOSPI 200_MA5'].shift(1) <= data_middle['KOSPI 200_MA15'].shift(1)) & (data_middle['KOSPI 200_MA5'] > data_middle['KOSPI 200_MA15'])
sell_cond = (data_middle['KOSPI 200_MA5'].shift(1) >= data_middle['KOSPI 200_MA15'].shift(1)) & (data_middle['KOSPI 200_MA5'] < data_middle['KOSPI 200_MA15'])

data_middle.loc[buy_cond, 'Signal1'] = 'Buy'
data_middle.loc[sell_cond, 'Signal1'] = 'Sell'

# Signal2: EMA5와 EMA15의 골든크로스와 데드크로스
data_middle['Signal2'] = 'Stay'

buy_cond = (data_middle['KOSPI 200_EMA5'].shift(1) <= data_middle['KOSPI 200_EMA15'].shift(1)) & (data_middle['KOSPI 200_EMA5'] > data_middle['KOSPI 200_EMA15'])
sell_cond = (data_middle['KOSPI 200_EMA5'].shift(1) >= data_middle['KOSPI 200_EMA15'].shift(1)) & (data_middle['KOSPI 200_EMA5'] < data_middle['KOSPI 200_EMA15'])

data_middle.loc[buy_cond, 'Signal2'] = 'Buy'
data_middle.loc[sell_cond, 'Signal2'] = 'Sell'

# Signal3: RSI14의 과매도와 과매수 신호에 따른 매매 신호
data_middle['Signal3'] = 'Stay'

buy_cond = (data_middle['KOSPI 200_RSI14'].shift(1) <= 30) & (data_middle['KOSPI 200_RSI14'] > 30)
sell_cond = (data_middle['KOSPI 200_RSI14'].shift(1) >= 70) & (data_middle['KOSPI 200_RSI14'] < 70)

data_middle.loc[buy_cond, 'Signal3'] = 'Buy'
data_middle.loc[sell_cond, 'Signal3'] = 'Sell'

# Signal4: Bollinger Bands 기반의 매매 신호
data_middle['Signal4'] = 'Stay'

buy_cond = (data_middle['KOSPI 200 Close'].shift(1) <= data_middle['KOSPI 200_BB_LOWER15'].shift(1)) & (data_middle['KOSPI 200 Close'] > data_middle['KOSPI 200_BB_LOWER15'])
sell_cond = (data_middle['KOSPI 200 Close'].shift(1) >= data_middle['KOSPI 200_BB_UPPER15'].shift(1)) & (data_middle['KOSPI 200 Close'] < data_middle['KOSPI 200_BB_UPPER15'])

data_middle.loc[buy_cond, 'Signal4'] = 'Buy'
data_middle.loc[sell_cond, 'Signal4'] = 'Sell'

In [33]:
# 각 신호 분포를 가로 테이블로 보기
signal_cols = ['Signal1', 'Signal2', 'Signal3', 'Signal4']

table = (
    pd.DataFrame({
        col: data_middle[col].value_counts(dropna=False)
        for col in signal_cols
    })
    .fillna(0)
    .astype(int)
    .reindex(['Stay', 'Buy', 'Sell'])   # 행 순서 고정
)

display(table)

,Signal1,Signal2,Signal3,Signal4
Stay,3797,3811,3855,3869
Buy,156,149,98,131
Sell,156,149,156,109


In [34]:
signal_cols = ['Signal1', 'Signal2', 'Signal3', 'Signal4']

# 1. 각 Signal의 범주 순서를 지정
for col in signal_cols:
    data_middle[col] = pd.Categorical(
        data_middle[col],
        categories=['Stay', 'Buy', 'Sell']
    )

# 2. Stay를 기준범주로 하여 가변수 생성
signal_dummies = pd.get_dummies(
    data_middle[signal_cols],
    prefix=signal_cols,
    prefix_sep='_',
    drop_first=True,
    dtype=int
)

# 3. 원본 데이터프레임에 붙이기
data_middle = pd.concat([data_middle, signal_dummies], axis=1)

data_middle = data_middle.drop(columns=['Signal1', 'Signal2', 'Signal3', 'Signal4'])

cols = [col for col in data_middle.columns if col != 'Risk_Label'] + ['Risk_Label']
data_middle = data_middle[cols]

print(data_middle.head())
print(list(data_middle.columns))

            Shanghai Comp  KODEX 200  TOPIX  Brent Crude Oil   Gold Spot  \
Date                                                                       
2009-04-17    2503.935059    17370.0  875.0        51.959999  867.400024   
2009-04-20    2557.456055    17480.0  876.0        51.959999  887.000000   
2009-04-21    2535.827881    17480.0  855.0        51.959999  882.099976   
2009-04-22    2461.345947    17715.0  856.0        51.959999  891.799988   
2009-04-23    2463.954102    17895.0  862.0        51.959999  905.900024   

            JPY/KRW      USD/KRW       NASDAQ      KOSDAQ  KOSPI 200 Close  \
Date                                                                         
2009-04-17   13.371  1325.800049  1673.069946  483.799988       171.330002   
2009-04-20   13.536  1327.500000  1608.209961  491.940002       172.300003   
2009-04-21   13.727  1354.300049  1643.849976  497.190002       171.960007   
2009-04-22   13.726  1346.599976  1646.119995  509.899994       174.399994   

In [35]:
print(data_middle.columns)

Index(['Shanghai Comp', 'KODEX 200', 'TOPIX', 'Brent Crude Oil', 'Gold Spot',
       'JPY/KRW', 'USD/KRW', 'NASDAQ', 'KOSDAQ', 'KOSPI 200 Close',
       'KOSPI 200 Open', 'KOSPI 200 High', 'KOSPI 200 Low', 'KOSPI 200 Volume',
       'CNY/KRW', 'KOSPI 200 Return', 'KOSPI 200 lagged_return_1',
       'KOSPI 200 lagged_return_2', 'KOSPI 200_MA5', 'KOSPI 200_MA15',
       'KOSPI 200_EMA5', 'KOSPI 200_EMA15', 'KOSPI 200_RSI14',
       'KOSPI 200_BB_MID15', 'KOSPI 200_BB_UPPER15', 'KOSPI 200_BB_LOWER15',
       'KOSPI 200_PDI14', 'KOSPI 200_MDI14', 'KOSPI 200_DMI14',
       'KOSPI 200_ADX14', 'KOSPI 200_ROC12', 'GARCH_mu_t1', 'GARCH_sigma_t1',
       'GARCH_VaR_5_t1', 'GJR_sigma_t1', 'GJR_VaR_5_t1', 'return(%)',
       'Signal1_Buy', 'Signal1_Sell', 'Signal2_Buy', 'Signal2_Sell',
       'Signal3_Buy', 'Signal3_Sell', 'Signal4_Buy', 'Signal4_Sell',
       'Risk_Label'],
      dtype='object')


In [36]:

#data_middle의 lagged_return 컬럼 %로 바꾸기
data_middle['KOSPI 200 lagged_return(%)_1'] = data_middle['KOSPI 200 lagged_return_1'] * 100
data_middle['KOSPI 200 lagged_return(%)_2'] = data_middle['KOSPI 200 lagged_return_2'] * 100
data_middle=data_middle.drop(columns=['KOSPI 200 Return', 'KOSPI 200 lagged_return_1', 'KOSPI 200 lagged_return_2'])

In [37]:
data_middle.columns

Index(['Shanghai Comp', 'KODEX 200', 'TOPIX', 'Brent Crude Oil', 'Gold Spot',
       'JPY/KRW', 'USD/KRW', 'NASDAQ', 'KOSDAQ', 'KOSPI 200 Close',
       'KOSPI 200 Open', 'KOSPI 200 High', 'KOSPI 200 Low', 'KOSPI 200 Volume',
       'CNY/KRW', 'KOSPI 200_MA5', 'KOSPI 200_MA15', 'KOSPI 200_EMA5',
       'KOSPI 200_EMA15', 'KOSPI 200_RSI14', 'KOSPI 200_BB_MID15',
       'KOSPI 200_BB_UPPER15', 'KOSPI 200_BB_LOWER15', 'KOSPI 200_PDI14',
       'KOSPI 200_MDI14', 'KOSPI 200_DMI14', 'KOSPI 200_ADX14',
       'KOSPI 200_ROC12', 'GARCH_mu_t1', 'GARCH_sigma_t1', 'GARCH_VaR_5_t1',
       'GJR_sigma_t1', 'GJR_VaR_5_t1', 'return(%)', 'Signal1_Buy',
       'Signal1_Sell', 'Signal2_Buy', 'Signal2_Sell', 'Signal3_Buy',
       'Signal3_Sell', 'Signal4_Buy', 'Signal4_Sell', 'Risk_Label',
       'KOSPI 200 lagged_return(%)_1', 'KOSPI 200 lagged_return(%)_2'],
      dtype='object')

In [38]:
# 컬럼 재배치
new_order = [
    'Shanghai Comp', 'KODEX 200', 'TOPIX', 'NASDAQ', 'KOSDAQ',
    'Brent Crude Oil', 'Gold Spot',
    'JPY/KRW', 'USD/KRW', 'CNY/KRW',
    'KOSPI 200 Close', 'KOSPI 200 Open', 'KOSPI 200 High', 'KOSPI 200 Low','KOSPI 200 Volume',
    'return(%)','KOSPI 200 lagged_return(%)_1', 'KOSPI 200 lagged_return(%)_2',
    'KOSPI 200_MA5', 'KOSPI 200_MA15', 'KOSPI 200_EMA5', 'KOSPI 200_EMA15',
    'KOSPI 200_RSI14', 'KOSPI 200_BB_MID15', 'KOSPI 200_BB_UPPER15', 'KOSPI 200_BB_LOWER15',
    'KOSPI 200_PDI14', 'KOSPI 200_MDI14', 'KOSPI 200_DMI14','KOSPI 200_ADX14', 'KOSPI 200_ROC12',
    'GARCH_mu_t1', 'GARCH_sigma_t1', 'GJR_sigma_t1', 'GARCH_VaR_5_t1', 'GJR_VaR_5_t1',
    'Signal1_Buy', 'Signal1_Sell', 'Signal2_Buy', 'Signal2_Sell',
    'Signal3_Buy', 'Signal3_Sell', 'Signal4_Buy', 'Signal4_Sell',
    'Risk_Label'
]

# 혹시 누락/오타 체크
missing_cols = [col for col in new_order if col not in data_middle.columns]
extra_cols = [col for col in data_middle.columns if col not in new_order]

print("누락된 컬럼:", missing_cols)
print("추가 컬럼:", extra_cols)

# 재배치
data_middle = data_middle[new_order]

# 5. 확인
print(data_middle.columns)
print(data_middle.shape)

누락된 컬럼: []
추가 컬럼: []
Index(['Shanghai Comp', 'KODEX 200', 'TOPIX', 'NASDAQ', 'KOSDAQ',
       'Brent Crude Oil', 'Gold Spot', 'JPY/KRW', 'USD/KRW', 'CNY/KRW',
       'KOSPI 200 Close', 'KOSPI 200 Open', 'KOSPI 200 High', 'KOSPI 200 Low',
       'KOSPI 200 Volume', 'return(%)', 'KOSPI 200 lagged_return(%)_1',
       'KOSPI 200 lagged_return(%)_2', 'KOSPI 200_MA5', 'KOSPI 200_MA15',
       'KOSPI 200_EMA5', 'KOSPI 200_EMA15', 'KOSPI 200_RSI14',
       'KOSPI 200_BB_MID15', 'KOSPI 200_BB_UPPER15', 'KOSPI 200_BB_LOWER15',
       'KOSPI 200_PDI14', 'KOSPI 200_MDI14', 'KOSPI 200_DMI14',
       'KOSPI 200_ADX14', 'KOSPI 200_ROC12', 'GARCH_mu_t1', 'GARCH_sigma_t1',
       'GJR_sigma_t1', 'GARCH_VaR_5_t1', 'GJR_VaR_5_t1', 'Signal1_Buy',
       'Signal1_Sell', 'Signal2_Buy', 'Signal2_Sell', 'Signal3_Buy',
       'Signal3_Sell', 'Signal4_Buy', 'Signal4_Sell', 'Risk_Label'],
      dtype='object')
(4109, 45)


In [39]:
data_middle.head()

,Shanghai Comp,KODEX 200,TOPIX,NASDAQ,KOSDAQ,Brent Crude Oil,Gold Spot,JPY/KRW,USD/KRW,CNY/KRW,...,GJR_VaR_5_t1,Signal1_Buy,Signal1_Sell,Signal2_Buy,Signal2_Sell,Signal3_Buy,Signal3_Sell,Signal4_Buy,Signal4_Sell,Risk_Label
Date,,,,,,,,,,,,,,,,,,,,,
2009-04-17,2503.935059,17370.0,875.0,1673.069946,483.799988,51.959999,867.400024,13.371,1325.800049,194.324755,...,-2.736404,0,0,0,0,0,0,0,0,Low Risk
2009-04-20,2557.456055,17480.0,876.0,1608.209961,491.940002,51.959999,887.000000,13.536,1327.500000,194.562510,...,-2.641002,0,0,0,0,0,0,0,0,Low Risk
2009-04-21,2535.827881,17480.0,855.0,1643.849976,497.190002,51.959999,882.099976,13.727,1354.300049,198.668030,...,-2.553085,0,0,0,0,0,0,0,0,Low Risk
2009-04-22,2461.345947,17715.0,856.0,1646.119995,509.899994,51.959999,891.799988,13.726,1346.599976,197.463154,...,-2.469281,0,0,0,0,0,0,0,0,Low Risk
2009-04-23,2463.954102,17895.0,862.0,1652.209961,514.090027,51.959999,905.900024,13.618,1333.599976,195.562586,...,-2.386329,0,0,0,0,0,1,0,0,Low Risk


In [40]:
data_middle.to_csv("../../data/processed/data_final.csv", index=True)

In [41]:
#data_middel label칼럼 shift(-1)하고 마지막 행 drop하기
data_middle_shift = data_middle.copy()
data_middle_shift['Risk_Label'] = data_middle_shift['Risk_Label'].shift(-1)


In [42]:
data_middle_shift.tail()

,Shanghai Comp,KODEX 200,TOPIX,NASDAQ,KOSDAQ,Brent Crude Oil,Gold Spot,JPY/KRW,USD/KRW,CNY/KRW,...,GJR_VaR_5_t1,Signal1_Buy,Signal1_Sell,Signal2_Buy,Signal2_Sell,Signal3_Buy,Signal3_Sell,Signal4_Buy,Signal4_Sell,Risk_Label
Date,,,,,,,,,,,,,,,,,,,,,
2026-02-23,4082.072998,86940.0,4001.0,22627.269531,1151.989990,71.489998,5204.700195,9.356865,1443.439941,208.951932,...,-3.323919,0,0,0,0,0,0,0,1,Low Risk
2026-02-24,4117.409180,89085.0,4011.0,22863.679688,1165.000000,70.769997,5155.799805,9.324820,1442.199951,208.772431,...,-3.252566,0,0,0,0,0,0,0,0,Low Risk
2026-02-25,4147.229980,90890.0,4045.0,23152.080078,1165.250000,70.849998,5206.399902,9.237456,1439.989990,209.206605,...,-3.155541,0,0,0,0,0,0,0,0,Low Risk
2026-02-26,4146.630859,95025.0,4078.0,22878.380859,1188.150024,70.750000,5176.500000,9.134690,1426.930054,207.728703,...,-3.266903,0,0,0,0,0,0,0,0,Low Risk
2026-02-27,4162.880859,94120.0,4131.0,22668.210938,1192.780029,72.480003,5230.500000,9.190693,1432.319946,209.375953,...,-3.187511,0,0,0,0,0,0,0,1,None


In [43]:
data_middle_shift = data_middle_shift.dropna(subset=['Risk_Label'])


In [44]:
data_middle_shift.to_csv("../../data/processed/data_final_shift.csv", index=True)